p

In [ ]:
print("hello world")

In [ ]:
try :
    import numba
    # Use for CPU,GPU Kaggle machines:
    print('Install CayleyPy without dependencies (for Kaggle CPU,GPU):'); print()
    !pip install git+https://github.com/cayleypy/cayleypy --no-deps   
except :
    print('Install CayleyPy with dependencies (for Kaggle TPU):'); print()
    # Use for TPU kaggle machines - since no numba
    !pip install git+https://github.com/cayleypy/cayleypy
    


Install CayleyPy without dependencies (for Kaggle CPU,GPU):

  Cloning https://github.com/cayleypy/cayleypy to c:\users\merav\appdata\local\temp\pip-req-build-k8onittv
  Resolved https://github.com/cayleypy/cayleypy to commit 6298ae1d9d63fd6a48a6bd67517aacf98633805d
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/cayleypy/cayleypy 'C:\Users\merav\AppData\Local\Temp\pip-req-build-k8onittv'

[notice] A new release of pip is available: 25.0.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install torch

import torch
import numpy as np 
import pandas as pd
torch.__version__

%%time 




[notice] A new release of pip is available: 25.0.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip
UsageError: Line magic function `%%time` not found.


In [3]:
from sympy.combinatorics import Permutation, PermutationGroup
### Finding Cayley graph size with Schreier Sims alogrithms ###

def do_schreier_sims(gens):
    pgens = [Permutation(list(q)) for q in gens]
    G     = PermutationGroup(*pgens)
    return G.order()



In [12]:
import os
import networkx as nx
from cayleypy import CayleyGraph, PermutationGroups
import re
import csv
import time
import json

### Saving to csv file
results_dir = "./results"
os.makedirs(results_dir, exist_ok=True)

csv_filename = os.path.join(results_dir, "koltsov3_fullgraph_perm1_d2_results.csv")
# Create file + header if it doesn't exist
write_header = not os.path.exists(csv_filename)

csv_file = open(csv_filename, "a", newline="")
csv_writer = csv.writer(csv_file)


if write_header:
    csv_writer.writerow([
        "k_param",
        # "d_param",
        "perm_type",
        "coset",
        "n_values",
        "diameters",
        "last_layers",
        "total_states",
        "num_n_computed",
    ])
    csv_file.flush()
csv_file.flush


<function TextIOWrapper.flush()>

In [ ]:
# %%time

max_n = 20
# diams = []
# dict_diams = {}
# last_layers = []

dict_growth = dict()
dict_last_layer = dict()


# ----------------------------
# Soft timeout settings
# ----------------------------
TIME_LIMIT_SEC = 30.0          # mark run as "slow" if BFS+SS exceeds this
STOP_ON_SLOW = True            # if True, stop the whole sweep once a slow run happens
MAX_SLOW_RUNS = 5              # if STOP_ON_SLOW is False, optionally stop after many slow runs
slow_runs = 0


#Koltsov graph params
# 3 generators:
# I generator swaps even elements - (0,1), (2,3),...
# K generator swaps odd elements - (1,2), (3,4),...
# S - special swap generator pattern controlled by params
perm_type = 1  # 1 - 1 transpostion (k, k + d), or 2 - product of 2 transpositions (k, k+3) and (k+1, k+2)
# k - starting index of S, window must be within size n (length of permutation)
d = 2         # distance of S, additional param for perm_type=1

# Graph definition:
# Part 1 -- generators: 
graph_name_part1 = 'Koltsov3'

# Part 2 -- Coset or not:
list_options = [' Coset', '']
graph_name_part2 = list_options[1]

    
graph_name = graph_name_part1+graph_name_part2



print('Graph:', graph_name )

for k in range(0,5):
    # for d in range(1,16):
    print('k = {k}, d = {d}'.format(k=k, d=d))

    # --- NEW: aggregators for this (k,d,perm_type) ---
    n_values = []
    diam_by_n = {}      # {n: diameter}
    states_by_n = {}    # {n: total_states}
    last_layers_by_n = {}  # {n: last_layer}

    start_n = k + d + 1
    if start_n > max_n:
        continue

    for n in range(10,20):
        if (n < start_n):
            print(f"  Skipping n={n} since it's less than k+d+1={start_n}")
            continue
        # for n in [36]:
        print('  n = {n}'.format(n=n))

        if 'Koltsov3' in graph_name:
            defn = PermutationGroups.koltsov3(n, perm_type=perm_type, k=k, d=d)
        else:
            print('Unknown type of generators')
            raise Exception("Unknown type of generators")
            
        t0 = time.time()
        graph = CayleyGraph(defn)
        total_states = do_schreier_sims(graph.generators)
        result = graph.bfs(return_all_edges=False, return_all_hashes=False)
        diameter = result.diameter()
        elapsed = time.time() - t0

        is_slow = (elapsed > TIME_LIMIT_SEC)

        if is_slow:
            slow_runs += 1
            print(f"⏱️ SLOW RUN: k={k}, d={d}, n={n} took {elapsed:.2f}s (limit {TIME_LIMIT_SEC}s)")

            if STOP_ON_SLOW:
                print("Stopping sweep due to STOP_ON_SLOW=True")
                raise SystemExit

            if slow_runs >= MAX_SLOW_RUNS:
                print(f"Stopping sweep after {slow_runs} slow runs (MAX_SLOW_RUNS={MAX_SLOW_RUNS})")
                raise SystemExit


        n_values.append(n)
        diam_by_n[n] = diameter
        states_by_n[n] = total_states
        last_layers_by_n[n] = len(result.last_layer())

        # Writing to CSV
        if 'Coset' in graph_name:
            coset_label = graph_name_part3.strip()  # e.g. "4Different"
        else:
            coset_label = "FullGraph"

        diameter = result.diameter()
        growth = result.layer_sizes
        last_layer_size = len(result.last_layer())

    # --- NEW: write exactly ONE row per (k,d,perm_type[,coset]) ---
    csv_writer.writerow([
        k,
        d,
        perm_type,
        coset_label,
        json.dumps(n_values),        # safer than str(list(...)) for CSV
        json.dumps(diam_by_n),       # dict {n:diam}
        json.dumps(last_layers_by_n),# dict {n:last_layer}
        json.dumps(states_by_n),     # dict {n:states}
        len(n_values)
    ])
    csv_file.flush()

csv_file.close()

Graph: Koltsov3
k = 0, d = 2
  n = 10


NameError: name 'PermutationGroups' is not defined